# Create BigQuery Table from SQL Script

This notebook executes the SQL DDL script and creates the BigQuery table.

Workflow:
1. Define project, dataset, table, and SQL script path
2. Create an authenticated BigQuery client
3. Read SQL from `ddl_scripts/sql_scripts/create_unmet_needs_table.sql`
4. Execute DDL in BigQuery
5. Validate table creation

## 1. Define Paths and Target Table

Set local SQL file path and BigQuery identifiers.

In [ ]:
from pathlib import Path
import os

PROJECT_ID = "cognito-prod-394707"
DATASET_ID = "cognito_prod_datamart"
TABLE_NAME = "unmet_needs_all_countries"
TABLE_ID = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME}"

# Notebook is under market_potential/unmet_needs/
BASE_DIR = Path.cwd()
SQL_FILE_PATH = BASE_DIR / "ddl_scripts" / "sql_scripts" / "create_unmet_needs_table.sql"

print(f"Target table: {TABLE_ID}")
print(f"SQL file path: {SQL_FILE_PATH}")
print(f"SQL file exists: {SQL_FILE_PATH.exists()}")

Target table: cognito-prod-394707.cognito_prod_datamart.unmet_needs_all_countries
SQL file path: c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\ddl_scripts\create_unmet_needs_table.sql
SQL file exists: True


## 2. Implement BigQuery Client

Use the provided helper function for credential-aware BigQuery authentication.

In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

def get_bq_client() -> bigquery.Client:
    """Get BigQuery client with credentials."""
    credentials_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

    if credentials_path and os.path.exists(credentials_path):
        credentials = service_account.Credentials.from_service_account_file(credentials_path)
        return bigquery.Client(project=PROJECT_ID, credentials=credentials)
    # Use default credentials (for Cloud Run attached service account)
    return bigquery.Client(project=PROJECT_ID)

client = get_bq_client()
print(f"Connected to BigQuery project: {client.project}")

Connected to BigQuery project: cognito-prod-394707


## 3. Read SQL DDL Script

Load SQL text from the DDL script file before execution.

In [3]:
if not SQL_FILE_PATH.exists():
    raise FileNotFoundError(f"DDL SQL file not found: {SQL_FILE_PATH}")

ddl_sql = SQL_FILE_PATH.read_text(encoding="utf-8").strip()

if not ddl_sql:
    raise ValueError("DDL SQL file is empty.")

print("DDL SQL loaded successfully.")
print("\n--- SQL PREVIEW (first 500 chars) ---")
print(ddl_sql[:500])

DDL SQL loaded successfully.

--- SQL PREVIEW (first 500 chars) ---
CREATE TABLE IF NOT EXISTS `cognito-prod-394707.cognito_prod_datamart.unmet_needs_all_countries` (
  unmet_need STRING OPTIONS (description = "Description of the unmet need in T2DM care"),
  category STRING OPTIONS (description = "Category of the unmet need"),
  evidence STRING OPTIONS (description = "Supporting evidence for the unmet need"),
  country STRING OPTIONS (description = "Country associated with the unmet need"),
  source STRING OPTIONS (description = "Data source for the unmet need r


## 4. Execute DDL in BigQuery

Run the SQL script to create the table.

In [4]:
ddl_job = client.query(ddl_sql)
result = ddl_job.result()

print("DDL executed successfully.")
print(f"Job ID: {ddl_job.job_id}")
print(f"Table create statement executed for: {TABLE_ID}")

DDL executed successfully.
Job ID: 716f9bd5-e1c1-472a-9263-1317e5ff6631
Table create statement executed for: cognito-prod-394707.cognito_prod_datamart.unmet_needs_all_countries


## 5. Validate Table Creation

Check metadata and confirm the table exists in the target dataset.

In [5]:
table = client.get_table(TABLE_ID)
print("Table found in BigQuery.")
print(f"Table ID: {table.full_table_id}")
print(f"Created: {table.created}")
print(f"Modified: {table.modified}")
print(f"Rows: {table.num_rows}")

print("\nSchema:")
for field in table.schema:
    print(f"- {field.name}: {field.field_type}")

Table found in BigQuery.
Table ID: cognito-prod-394707:cognito_prod_datamart.unmet_needs_all_countries
Created: 2026-06-13 15:28:07.944000+00:00
Modified: 2026-06-13 15:28:07.944000+00:00
Rows: 0

Schema:
- unmet_need: STRING
- category: STRING
- evidence: STRING
- country: STRING
- source: STRING
- created_at_utc: TIMESTAMP


## 6. Load CSV Data and Add created_at_utc

Read unmet needs CSV data, add a UTC timestamp column, and prepare records for BigQuery load.

In [11]:
import pandas as pd

CSV_PATH = BASE_DIR / "outputs" / "unmet_needs_all_countries_latest_final.xlsx"

if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")

df = pd.read_excel(CSV_PATH)

required_columns = ["unmet_need", "category", "evidence", "country", "source"]
missing_columns = [c for c in required_columns if c not in df.columns]
if missing_columns:
    raise ValueError(f"CSV missing required columns: {missing_columns}")

# Add UTC timestamp for load tracking
df["created_at_utc"] = pd.Timestamp.now(tz="UTC")

print(f"CSV loaded successfully from: {CSV_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print("\nSample rows:")
print(df.head(3).to_string(index=False))

CSV loaded successfully from: c:\Users\p00102977\Desktop\Projects\API\FInal Repo content\drl-cognito-ai\market_potential\unmet_needs\outputs\unmet_needs_all_countries_latest_final.xlsx
Rows: 224
Columns: ['unmet_need', 'category', 'evidence', 'country', 'source', 'created_at_utc']

Sample rows:
                                                                                                              unmet_need                        category                                                                                                                                                                                     evidence country       source                   created_at_utc
                             Nearly half of U.S. adults with diabetes fail to achieve the recommended A1C target of <7%. Achievement of Clinical Targets In 2015–2018 (the most recent time period with population-level data available), just 50.5% of U.S. community-dwelling adults with diabetes achieved A1C <7

## 7. Load DataFrame into BigQuery Table

Append CSV rows into the table using explicit schema and WRITE_APPEND mode.

In [12]:
load_schema = [
    bigquery.SchemaField("unmet_need", "STRING"),
    bigquery.SchemaField("category", "STRING"),
    bigquery.SchemaField("evidence", "STRING"),
    bigquery.SchemaField("country", "STRING"),
    bigquery.SchemaField("source", "STRING"),
    bigquery.SchemaField("created_at_utc", "TIMESTAMP"),
]

job_config = bigquery.LoadJobConfig(
    schema=load_schema,
    write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
)

load_job = client.load_table_from_dataframe(
    df,
    TABLE_ID,
    job_config=job_config,
)
load_job.result()

print("Data load completed.")
print(f"Job ID: {load_job.job_id}")
print(f"Rows appended: {load_job.output_rows}")

Data load completed.
Job ID: 95791b0c-4fd7-413f-96bd-88ee8acdfd9e
Rows appended: 224


## 8. Post-Load Validation

Validate final table row count and inspect latest inserted rows.

In [13]:
summary_query = f"""
SELECT
  COUNT(*) AS total_rows,
  MIN(created_at_utc) AS min_created_at_utc,
  MAX(created_at_utc) AS max_created_at_utc
FROM `{TABLE_ID}`
"""

summary_df = client.query(summary_query).result().to_dataframe()
print("Table summary:")
print(summary_df.to_string(index=False))

latest_rows_query = f"""
SELECT unmet_need, category, country, source, created_at_utc
FROM `{TABLE_ID}`
ORDER BY created_at_utc DESC
LIMIT 10
"""

latest_rows_df = client.query(latest_rows_query).result().to_dataframe()
print("\nLatest 10 rows:")
print(latest_rows_df.to_string(index=False))

print(f"\nCSV row count loaded in this run: {len(df)}")

Table summary:
 total_rows               min_created_at_utc               max_created_at_utc
        224 2026-06-13 15:44:12.883572+00:00 2026-06-13 15:44:12.883572+00:00

Latest 10 rows:
                                                                                                                                        unmet_need                              category country       source                   created_at_utc
                                               The current healthcare delivery system is fragmented and poorly designed for chronic diabetes care. Access, Affordability & Care Delivery      US soc_document 2026-06-13 15:44:12.883572+00:00
                                                          A significant percentage of adults with diabetes ration their insulin due to high costs. Access, Affordability & Care Delivery      US soc_document 2026-06-13 15:44:12.883572+00:00
                                  High-deductible health plans increase financial hardship and 